# GeoTIFF viewer

View any `.tif` / `.tiff` from this project: raw Sentinel-5P, raw Sentinel-2, or the Jan–May 2025 forecast files.

**How to use**
1. Set `IMAGE_PATH` (and optionally the other settings) in the **settings** cell below.
2. Run all cells (*Run All*).

Paths are relative to the `Scratch/` folder; absolute paths also work. The last cell lists every `.tif` in the project
so you can copy one.

In [ ]:
# ============================== SETTINGS: edit these ==============================
IMAGE_PATH = "/home/naresh/Projects/FINAL YEAR/Scratch/Forecasting/results/F5_final/outlook_2025/outlook_NO2_2025.tif"
# Examples:
# IMAGE_PATH = "Dataset/data/s5p_composites/s5p_20200301.tif"                        # Sentinel-5P: NO2, CO, SO2 (13x13, ~5 km)
# IMAGE_PATH = "Dataset/data/s2_composites/s2_20200301.tif"                          # Sentinel-2: 12 bands (558x558, 100 m)
# IMAGE_PATH = "Forecasting/results/F5_final/outlook_2025/outlook_NO2_2025.tif"      # 2025 forecast: 90 bands

BANDS = None      # None = first 12 bands; or a list of 1-based band numbers, e.g. [1, 6, 12, 18, 24, 30]
RGB = False       # True = Sentinel-2 true-colour photo (red/green/blue bands)
VMIN = None       # colour-scale limits; None = automatic (2nd-98th percentile of each band)
VMAX = None
SAVE_TO = None    # None = show only; or a PNG path, e.g. "my_view.png"
# ====================================================================================

In [ ]:
%matplotlib inline
import logging
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio

# GDAL warns about a harmless header quirk in the Sentinel-2 files; the pixel data is fine
logging.getLogger("rasterio").setLevel(logging.ERROR)

def project_root() -> Path:
    """The Scratch/ folder: the nearest parent of the working directory that holds Dataset/."""
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "Dataset").is_dir() and (d / "pyproject.toml").exists():
            return d
    return here

ROOT = project_root()
path = Path(IMAGE_PATH)
path = path if path.is_absolute() else ROOT / path
assert path.exists(), f"File not found: {path}"
print("Viewing:", path)

In [ ]:
S2_NAMES = ["B2 blue", "B3 green", "B4 red", "B5 red-edge 1", "B6 red-edge 2", "B7 red-edge 3", "B8 NIR",
            "B11 SWIR-1", "B12 SWIR-2", "NDVI", "NDBI", "NDMI"]
S5P_NAMES = ["NO2 (mol/m²)", "CO (mol/m²)", "SO2 (mol/m²)"]


def band_names(src):
    """The file's own band descriptions if present, else the known S5P / S2 layout, else 'band N'."""
    desc = list(src.descriptions)
    if any(desc):
        return [d or f"band {i + 1}" for i, d in enumerate(desc)]
    name = Path(src.name).name
    if src.count == 3 and name.startswith("s5p"):
        return S5P_NAMES
    if src.count == 12 and name.startswith("s2"):
        return S2_NAMES
    return [f"band {i + 1}" for i in range(src.count)]


def read(src, bands):
    """Read bands (1-based) as float, with nodata (-inf, NaN or the file's nodata value) set to NaN."""
    a = src.read(bands).astype("float64")
    a[~np.isfinite(a)] = np.nan
    if src.nodata is not None and np.isfinite(src.nodata):
        a[a == src.nodata] = np.nan
    return a

In [ ]:
# Metadata and per-band statistics
with rasterio.open(path) as src:
    names = band_names(src)
    b = src.bounds
    print(f"size   : {src.count} band(s) x {src.height} rows x {src.width} cols, {src.dtypes[0]}")
    print(f"CRS    : {src.crs}   pixel size: {src.res[0]:.5f} x {src.res[1]:.5f} degrees")
    print(f"extent : lon {b.left:.3f} to {b.right:.3f}, lat {b.bottom:.3f} to {b.top:.3f}")
    tags = {k: v for k, v in src.tags().items() if not k.startswith(("TIFFTAG", "AREA"))}
    if tags:
        print(f"tags   : {tags}")
    print()
    for i in range(1, src.count + 1):
        a = read(src, [i])[0]
        ok = np.isfinite(a)
        stats = f"min {np.nanmin(a):.4g}  mean {np.nanmean(a):.4g}  max {np.nanmax(a):.4g}" if ok.any() else "all nodata"
        model = src.tags(i).get("model")
        print(f"{i:>3}  {names[i - 1]:<40} valid {ok.mean():6.1%}  {stats}" + (f"  [model: {model}]" if model else ""))

In [ ]:
# Plot: a band grid (grey = no data), or a Sentinel-2 true-colour image when RGB = True
with rasterio.open(path) as src:
    names = band_names(src)
    b = src.bounds
    extent = [b.left, b.right, b.bottom, b.top]
    if RGB:
        assert src.count >= 3, "RGB needs a multi-band (Sentinel-2) file"
        idx = [3, 2, 1] if src.count == 12 else [1, 2, 3]          # S2 layout: B4 red, B3 green, B2 blue
        a = read(src, idx)
        lo, hi = np.nanpercentile(a, [2, 98])
        img = np.clip((a - lo) / (hi - lo), 0, 1).transpose(1, 2, 0)
        img = np.where(np.isfinite(img), img, 0.75)                 # no data in grey
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(img, extent=extent)
        ax.set_title(f"{path.name}: true colour (bands {idx})")
        ax.set_xlabel("longitude (°E)")
        ax.set_ylabel("latitude (°N)")
    else:
        bands = BANDS or list(range(1, min(src.count, 12) + 1))
        bad = [x for x in bands if not 1 <= x <= src.count]
        assert not bad, f"band(s) {bad} out of range 1..{src.count}"
        if BANDS is None and src.count > 12:
            print(f"Showing bands 1-12 of {src.count}; set BANDS to choose others.")
        cols = min(len(bands), 4)
        rows = math.ceil(len(bands) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(4.4 * cols, 4.0 * rows), squeeze=False)
        cmap = plt.get_cmap("viridis").copy()
        cmap.set_bad("#bdbdbd")
        for k, ax in enumerate(axes.ravel()):
            if k >= len(bands):
                ax.axis("off")
                continue
            a = read(src, [bands[k]])[0]
            lo, hi = (np.nanpercentile(a, [2, 98]) if np.isfinite(a).any() else (None, None))
            im = ax.imshow(np.ma.masked_invalid(a), extent=extent, cmap=cmap, interpolation="nearest",
                           vmin=lo if VMIN is None else VMIN, vmax=hi if VMAX is None else VMAX)
            ax.set_title(f"{bands[k]}: {names[bands[k] - 1]}", fontsize=8)
            ax.tick_params(labelsize=7)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).ax.tick_params(labelsize=7)
        fig.suptitle(path.name)
        fig.tight_layout()
    if SAVE_TO:
        out = Path(SAVE_TO) if Path(SAVE_TO).is_absolute() else ROOT / SAVE_TO
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print("saved", out)
    plt.show()

### Optional: list every `.tif` in the project (copy a path into `IMAGE_PATH`)

In [ ]:
groups = {"Sentinel-5P": "Dataset/data/s5p_composites", "Sentinel-2": "Dataset/data/s2_composites",
          "Forecast outputs": "Forecasting/results"}
for label, folder in groups.items():
    files = [f.relative_to(ROOT) for f in sorted((ROOT / folder).rglob("*.tif"))]
    print(f"{label}: {len(files)} file(s) in {folder}/")
    shown = files if len(files) <= 10 else files[:5] + ["..."] + files[-5:]
    for f in shown:
        print("   ", f)